# Batch Segmentation with SAM2 — Local PC

Combined notebook: `SegmentationModule` class + batch processing pipeline.

**Expected repo layout on your machine:**
```
CAPSTONE_PROJECT_REPO/
├── checkpoints/
│   └── sam2_hiera_large.pt
├── configs/
│   └── sam2/
│       └── sam2_hiera_l.yaml
├── SAMPLE_INPUT/
│   ├── image1.jpg
│   └── ...
└── SAMPLE_OUTPUT/       ← created automatically
```

## Configuration
Edit the paths below if needed. Everything else runs as-is.

In [16]:
import os

# =============================
# CONFIGURATION
# =============================
REPO_DIR   = r"D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO"
INPUT_DIR  = os.path.join(REPO_DIR, "capstone-demonstration-inputs")
OUTPUT_DIR = os.path.join(REPO_DIR, "capstone-demonstration-outputs")

# Checkpoint & config paths (relative to REPO_DIR, matching segmentation_module.py)
CHECKPOINT   = os.path.join(REPO_DIR,"food-calorie-estimator", "designing_nd_testing","segmentation_module", "checkpoints", "sam2_hiera_large.pt")
MODEL_CONFIG  = "configs/sam2/sam2_hiera_l.yaml"

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png")

# Output sub-directories
COORDS_DIR   = os.path.join(OUTPUT_DIR, "coordinates")
MASKS_DIR    = os.path.join(OUTPUT_DIR, "masks")
SEGMENTS_DIR = os.path.join(OUTPUT_DIR, "segments")

for d in [COORDS_DIR, MASKS_DIR, SEGMENTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Repo    : {REPO_DIR}")
print(f"Input   : {INPUT_DIR}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Checkpoint exists: {os.path.exists(CHECKPOINT)}")

Repo    : D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO
Input   : D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\capstone-demonstration-inputs
Output  : D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\capstone-demonstration-outputs
Checkpoint exists: True


## Imports

In [17]:
import json
import time
import cv2
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## SegmentationModule Class

In [18]:
class SegmentationModule:
    """
    A segmentation module for food image segmentation using SAM2.
    """

    def __init__(self, checkpoint=CHECKPOINT, model_config=MODEL_CONFIG, device=DEVICE):
        """
        Initialize the SegmentationModule.

        Args:
            checkpoint   (str): Path to the SAM2 .pt checkpoint file.
            model_config (str): SAM2 config path (e.g. 'configs/sam2/sam2_hiera_l.yaml').
            device       (str): 'cuda' or 'cpu'.
        """
        self.checkpoint   = checkpoint
        self.model_config = model_config
        self.device       = device

        # SAM2 mask-generator parameters
        self.points_per_side        = 64
        self.points_per_batch       = 32
        self.pred_iou_thresh        = 0.8
        self.stability_score_thresh = 0.92
        self.box_nms_thresh         = 0.6
        self.min_mask_region_area   = 2000
        self.dedup_iou_threshold    = 0.85
        self.min_area_ratio         = 0.20 # Only keep masks that are at least this fraction of the largest mask's area

        self.mask_generator = self._load_model()

    # ------------------------------------------------------------------
    # PRIVATE: Model loading
    # ------------------------------------------------------------------

    def _load_model(self):
        if not os.path.exists(self.checkpoint):
            raise FileNotFoundError(f"SAM2 checkpoint not found at {self.checkpoint}")

        print(f"Loading SAM2 on {self.device}...")
        sam2 = build_sam2(
            self.model_config, self.checkpoint,
            device=self.device, apply_postprocessing=True
        )
        return SAM2AutomaticMaskGenerator(
            model=sam2,
            points_per_side=self.points_per_side,
            points_per_batch=self.points_per_batch,
            pred_iou_thresh=self.pred_iou_thresh,
            stability_score_thresh=self.stability_score_thresh,
            box_nms_thresh=self.box_nms_thresh,
            min_mask_region_area=self.min_mask_region_area,
        )

    # ------------------------------------------------------------------
    # PRIVATE: Helpers
    # ------------------------------------------------------------------

    def _load_image_rgb(self, input):
        """
        Accept a file path (str) or a PIL Image and return an RGB numpy array.
        """
        if isinstance(input, str):
            if not os.path.exists(input):
                raise FileNotFoundError(f"Image not found at {input}")
            image = cv2.imdecode(np.fromfile(input, dtype=np.uint8), cv2.IMREAD_COLOR)
            if image is None:
                raise ValueError(f"Failed to read image: {input}")
            return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        elif isinstance(input, Image.Image):
            return np.array(input.convert("RGB"))
        else:
            raise TypeError("Input must be a file path (str) or a PIL.Image object")

    def _generate_masks(self, image_rgb):
        """Run SAM2 and apply area-based filtering + deduplication."""
        masks = self.mask_generator.generate(image_rgb)
        masks = self._deduplicate_masks(masks)
        if masks:
            max_area = max(m["area"] for m in masks)
            masks = [m for m in masks if m["area"] >= self.min_area_ratio * max_area]
        return masks

    def _deduplicate_masks(self, masks):
        """Remove heavily overlapping masks, keeping the larger one."""
        masks = sorted(masks, key=lambda m: m["area"], reverse=True)
        kept = []
        for candidate in masks:
            c_mask = candidate["segmentation"]
            is_duplicate = False
            for kept_mask in kept:
                k_mask = kept_mask["segmentation"]
                intersection = np.logical_and(c_mask, k_mask).sum()
                union        = np.logical_or(c_mask,  k_mask).sum()
                if union > 0 and (intersection / union) > self.dedup_iou_threshold:
                    is_duplicate = True
                    break
            if not is_duplicate:
                kept.append(candidate)
        return kept

    # ------------------------------------------------------------------
    # PUBLIC: Input methods
    # ------------------------------------------------------------------

    def set_image_from_path(self, image_path):
        """Load and cache an image from a file path."""
        self._cached_image_rgb = self._load_image_rgb(image_path)
        self._cached_masks = None

    def set_image(self, image):
        """Load and cache a PIL Image."""
        self._cached_image_rgb = self._load_image_rgb(image)
        self._cached_masks = None

    def _get_cached_masks(self, input=None):
        """Return cached masks, generating them if not yet computed."""
        if input is not None:
            image_rgb = self._load_image_rgb(input)
            return image_rgb, self._generate_masks(image_rgb)

        if not hasattr(self, "_cached_image_rgb") or self._cached_image_rgb is None:
            raise RuntimeError(
                "No image loaded. Call set_image() or set_image_from_path() first, "
                "or pass an image directly to the method."
            )
        if self._cached_masks is None:
            self._cached_masks = self._generate_masks(self._cached_image_rgb)
        return self._cached_image_rgb, self._cached_masks

    # ------------------------------------------------------------------
    # PUBLIC: Output methods
    # ------------------------------------------------------------------

    def get_coordinates(self, input=None):
        """
        Return bounding-box coordinates for each detected segment.

        Returns list of dicts with keys:
            segment_id, bbox, area, predicted_iou, stability_score
        """
        _, masks = self._get_cached_masks(input)
        return [
            {
                "segment_id":      i,
                "bbox":            mask["bbox"],
                "area":            mask["area"],
                "predicted_iou":   round(mask["predicted_iou"],   4),
                "stability_score": round(mask["stability_score"], 4),
            }
            for i, mask in enumerate(masks)
        ]

    def get_masks(self, input=None):
        """
        Return binary boolean masks for each detected segment.

        Returns list of dicts with keys:
            segment_id, mask (np.ndarray bool H×W), area
        """
        _, masks = self._get_cached_masks(input)
        return [
            {
                "segment_id": i,
                "mask":       mask["segmentation"].astype(bool),
                "area":       mask["area"],
            }
            for i, mask in enumerate(masks)
        ]

    def get_segments(self, input=None):
        """
        Return cropped RGBA PIL images for each detected segment.
        Background outside the mask is transparent.

        Returns list of dicts with keys:
            segment_id, image (PIL RGBA tightly cropped), bbox, area
        """
        image_rgb, masks = self._get_cached_masks(input)
        results = []
        for i, mask in enumerate(masks):
            mask_bool = mask["segmentation"].astype(bool)

            rgba = np.zeros((*image_rgb.shape[:2], 4), dtype=np.uint8)
            rgba[..., :3] = image_rgb
            rgba[..., 3]  = (mask_bool * 255).astype(np.uint8)

            results.append({
                "segment_id": i,
                "image":      Image.fromarray(rgba, mode="RGBA"),  # full size, no crop
                "bbox":       mask["bbox"],
                "area":       mask["area"],
            })
        return results

    def clear_cache(self):
        """Release cached image and masks, and free GPU memory."""
        self._cached_image_rgb = None
        self._cached_masks     = None
        torch.cuda.empty_cache()

print("SegmentationModule class defined.")

SegmentationModule class defined.


## Instantiate the Module

In [19]:
seg = SegmentationModule()
seg.min_area_ratio = 0.02       # was 0.30 — the main fix
seg.min_mask_region_area = 100  # was 2000 — lets smaller food regions through
seg.dedup_iou_threshold = 0.5
print("Model loaded and ready.")

Loading SAM2 on cuda...
Model loaded and ready.


## Batch Processing Loop

For every image in `SAMPLE_INPUT` the pipeline:
1. Saves **coordinates** as JSON files
2. Saves **masks** as `.npy` files
3. Saves **segments** as transparent-background PNG files

In [20]:
image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
]

print(f"Found {len(image_files)} images in {INPUT_DIR}\n")

total_start = time.time()

for filename in image_files:
    image_path = os.path.join(INPUT_DIR, filename)
    base_name  = os.path.splitext(filename)[0]

    print(f"Processing: {filename}")
    t0 = time.time()

    # Per-image sub-folders
    img_coords_dir   = os.path.join(COORDS_DIR,   base_name)
    img_masks_dir    = os.path.join(MASKS_DIR,    base_name)
    img_segments_dir = os.path.join(SEGMENTS_DIR, base_name)
    for d in [img_coords_dir, img_masks_dir, img_segments_dir]:
        os.makedirs(d, exist_ok=True)

    # Load image once; results are cached for all three calls
    seg.set_image_from_path(image_path)

    # 1. COORDINATES → JSON
    coordinates = seg.get_coordinates()
    for coord in coordinates:
        sid      = coord["segment_id"]
        out_path = os.path.join(img_coords_dir, f"{base_name}_coord_{sid:03d}.json")
        with open(out_path, "w") as f:
            json.dump(coord, f, indent=4)
    print(f"  ✔ Coordinates saved  ({len(coordinates)} segments)")

    # 2. MASKS → NPY
    masks = seg.get_masks()
    for mask_data in masks:
        sid      = mask_data["segment_id"]
        out_path = os.path.join(img_masks_dir, f"{base_name}_mask_{sid:03d}.npy")
        np.save(out_path, mask_data["mask"])
        out_path = os.path.join(img_masks_dir, f"{base_name}_mask_{sid:03d}.txt")
        np.savetxt(out_path, mask_data["mask"].astype(int), fmt="%d")
    print(f"  ✔ Masks saved        ({len(masks)} segments)")

    # 3. SEGMENTS → PNG
    segments = seg.get_segments()
    for segment in segments:
        sid      = segment["segment_id"]
        out_path = os.path.join(img_segments_dir, f"{base_name}_segment_{sid:03d}.png")
        segment["image"].save(out_path)
    print(f"  ✔ Segments saved     ({len(segments)} segments)")

    elapsed = time.time() - t0
    print(f"  ⏱  {elapsed:.2f}s\n")

    # Free GPU memory before next image
    seg.clear_cache()

total_elapsed = time.time() - total_start
print(f"All done! Total time: {total_elapsed:.2f}s")
print(f"Output written to: {OUTPUT_DIR}")

Found 6 images in D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\capstone-demonstration-inputs

Processing: input_1_side.jpeg
  ✔ Coordinates saved  (7 segments)
  ✔ Masks saved        (7 segments)
  ✔ Segments saved     (7 segments)
  ⏱  50.99s

Processing: input_1_top.jpeg
  ✔ Coordinates saved  (6 segments)
  ✔ Masks saved        (6 segments)
  ✔ Segments saved     (6 segments)
  ⏱  47.74s

Processing: input_2_side.jpeg
  ✔ Coordinates saved  (8 segments)
  ✔ Masks saved        (8 segments)
  ✔ Segments saved     (8 segments)
  ⏱  46.82s

Processing: input_2_top.jpeg
  ✔ Coordinates saved  (3 segments)
  ✔ Masks saved        (3 segments)
  ✔ Segments saved     (3 segments)
  ⏱  32.72s

Processing: input_3_side.jpeg
  ✔ Coordinates saved  (15 segments)
  ✔ Masks saved        (15 segments)
  ✔ Segments saved     (15 segments)
  ⏱  35.55s

Processing: input_3_top.jpeg
  ✔ Coordinates saved  (6 segments)
  ✔ Masks saved        (6 segments)
  ✔ Segments saved     (6 segments)


## Output Structure
```
SAMPLE_OUTPUT/
├── coordinates/
│   └── <image_name>/
│       ├── <image_name>_coord_000.json
│       ├── <image_name>_coord_001.json
│       └── ...
├── masks/
│   └── <image_name>/
│       ├── <image_name>_mask_000.npy
│       ├── <image_name>_mask_001.npy
│       └── ...
└── segments/
    └── <image_name>/
        ├── <image_name>_segment_000.png
        ├── <image_name>_segment_001.png
        └── ...
```